# T8.4 — ước lượng lương, head Bi-GRU ‖ Bi-LSTM → CNN đọc token PhoBERT

Một run trên họ cột `masked` (che số lương — quy tắc chống rò rỉ). Cấu hình theo
Tran–Vo–Luu 2022: `--hidden 100 --conv-channels 50`. Trần **35 phút**; dự kiến
**~50 phút** cả notebook.

Mốc để so: `dl-sal-cpu-0920` MAE **4,13 triệu** · R²(log) 0,515.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi | head -15

In [ ]:
import os
if not os.path.isdir('vietjobs'):
    !git clone -q -b t8-rnn-colab https://github.com/boy2407/vietjobs.git
%cd vietjobs
!git log --oneline -1
# KHÔNG cài lại torch — giữ bản CUDA sẵn có của Colab (requirements-dl.txt nói rõ).
!pip install -q "transformers>=4.38,<4.47" "pyarrow>=15.0" "scikit-learn>=1.5" \
    "scipy>=1.13" "joblib>=1.4" "underthesea>=6.8" pytest
!pip install -q -e . --no-deps
%env PYTHONPATH=src

## Khôi phục từ Drive + cổng test

Mount Drive, lấy splits (337 MB) và cache nhỏ, encode lại **train token họ `masked`** (~8–10 phút T4), rồi `pytest` canh cache khớp split. Đỏ thì dừng — đừng huấn luyện.

In [ ]:
!python notebooks/colab_setup.py --family masked

## Hàm lưu kết quả — in F1 score, zip, lưu Drive, tải về

In [ ]:
import json, pathlib, shutil, subprocess
from google.colab import files

RUNS_ON_DRIVE = pathlib.Path('/content/drive/MyDrive/vietjobs_runs')
RUNS_ON_DRIVE.mkdir(parents=True, exist_ok=True)

def save_run(run_id):
    """In F1 score, rồi zip + lưu Drive + tải về máy — gọi NGAY sau mỗi run."""
    d = pathlib.Path('artifacts') / run_id
    mp = d / 'metrics.json'
    if not mp.exists():
        print(f'!! {run_id}: không có metrics.json — run chưa xong hoặc đã lỗi'); return
    m = json.loads(mp.read_text())
    met = m['metrics']
    print(f"\n=== {run_id} · {m['model']} · best epoch {m['best_epoch']}/{m['epochs_run']} "
          f"· dừng vì: {m.get('stopped_by')} · {m['seconds']/60:.1f} phút ===")
    if m.get('per_class'):
        pc = m['per_class']
        rows = sorted((k, v) for k, v in pc.items()
                      if k not in ('accuracy', 'macro avg', 'weighted avg'))
        rows.sort(key=lambda kv: -kv[1]['support'])
        print(f"{'ngành':40s} {'P':>6} {'R':>6} {'F1':>6} {'n':>5}")
        for k, v in rows:
            print(f"{k[:40]:40s} {v['precision']:6.3f} {v['recall']:6.3f} "
                  f"{v['f1-score']:6.3f} {int(v['support']):5d}")
        print('-' * 66)
        print(f"F1 score (macro)    = {met['f1_macro']:.4f}   <- trung bình cộng 16 F1 ở trên; "
              f"độ đo chọn mô hình")
        print(f"F1 score (weighted) = {met['f1_weighted']:.4f}   <- trung bình theo n")
        print(f"F1 score (micro)    = {met['f1_micro']:.4f}   <- = accuracy = F1 của bài Tran–Vo–Luu")
        print(f"accuracy            = {met['accuracy']:.4f}")
        if 'f1_macro_no_junk' in met:
            print(f"macro-F1 bỏ lớp nhóm_nghề_khác = {met['f1_macro_no_junk']:.4f}")
    else:
        print(f"MAE = {met['mae_trieu']:.2f} triệu · RMSE = {met['rmse_trieu']:.2f} · "
              f"R²(log) = {met['r2_log']:.3f} · ±20% = {met['within_20pct']*100:.1f}%")
    # dòng cuối của log kết quả — chính là dòng run này vừa ghi
    tail = subprocess.run(['tail', '-1', 'docs/04-results.md'], capture_output=True, text=True).stdout
    (d / '04-results.tail.md').write_text(tail)
    zip_path = shutil.make_archive(f'/content/{run_id}', 'zip', d)
    shutil.copy(zip_path, RUNS_ON_DRIVE / f'{run_id}.zip')
    with (RUNS_ON_DRIVE / '04-results.tail.md').open('a') as fh:
        fh.write(tail)
    print(f"đã lưu {RUNS_ON_DRIVE / (run_id + '.zip')} ({pathlib.Path(zip_path).stat().st_size/1e6:.1f} MB) — đang tải về máy…")
    files.download(zip_path)

## Huấn luyện — mỗi cell một run, `save_run` ngay sau

In [ ]:
!python -m vietjobs.dl.train_dl --task salary --head rnn --device cuda --hidden 100 --conv-channels 50 --max-minutes 35  --run-id dl-sal-rnn
save_run("dl-sal-rnn")

## Thời gian thật mỗi epoch trên T4

In [ ]:
import json, pathlib, glob
for hp in sorted(glob.glob('artifacts/dl-*rnn*/history.jsonl')):
    h = [json.loads(l) for l in open(hp)]
    if h:
        print(f"{pathlib.Path(hp).parent.name:22s} {len(h):3d} epoch · "
              f"{h[-1]['seconds']/len(h):5.0f} giây/epoch · tổng {h[-1]['seconds']/60:5.1f} phút")

## Mang về máy chính

Mỗi run đã được tải về ngay khi xong (`<run_id>.zip`), và cũng nằm ở
`MyDrive/vietjobs_runs/`. Về máy: giải nén vào `artifacts/<run_id>/`, rồi đưa tôi
các thư mục đó — tôi ghép đúng 1 dòng mới vào `docs/04-results.md` (chỉ-thêm),
điền bảng ở `docs/06` và chương 4 luận văn từ `metrics.json` thật.

`stopped_by: time` nghĩa là run bị trần 35 phút cắt trước khi hội tụ tự nhiên —
kết quả vẫn hợp lệ nhưng phải ghi rõ trong bảng.